<a href="https://colab.research.google.com/github/shridhartapakire/movie-recommendation-system/blob/main/movie_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System

A content-based movie recommendation system using Natural Language Processing (NLP), text vectorization, and cosine similarity.

## Project Goals
- Explore and clean movie data.
- Build a content-based recommendation engine.
- Create a recommendation API using FastAPI.
- Develop a simple user interface.
- Track development using Git and GitHub.

In [13]:
# Check the Python environment
import sys

print("Python version:", sys.version)
print("Movie Recommendation System")
print("Day 1: Data Gathering and Cleaning")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Movie Recommendation System
Day 1: Data Gathering and Cleaning


In [14]:
import pandas as pd

df = pd.read_csv("movies_metadata.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

df.head()

Dataset shape: (45466, 24)

Column names:
['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


/tmp/ipykernel_4527/2078642297.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("movies_metadata.csv")


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [15]:
# Reload the dataset with consistent type inference
df = pd.read_csv("movies_metadata.csv", low_memory=False)

# Inspect missing values
print("Missing values in important columns:")
print(df[["title", "overview", "genres", "id"]].isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check data types
print("\nData types:")
print(df[["title", "overview", "genres", "id", "popularity"]].dtypes)

# Preview movie titles and descriptions
df[["title", "overview"]].head(10)

Missing values in important columns:
title         6
overview    954
genres        0
id            0
dtype: int64

Duplicate rows: 17

Data types:
title         object
overview      object
genres        object
id            object
popularity    object
dtype: object


,title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...
5,Heat,"Obsessive master thief, Neil McCauley leads a ..."
6,Sabrina,An ugly duckling having undergone a remarkable...
7,Tom and Huck,"A mischievous young boy, Tom Sawyer, witnesses..."
8,Sudden Death,International action superstar Jean Claude Van...
9,GoldenEye,James Bond must unmask the mysterious head of ...


In [16]:
# Keep the original dataset unchanged
movies = df.copy()

# 1. Remove exact duplicate rows
movies = movies.drop_duplicates()

# 2. Remove rows with missing or blank titles
movies["title"] = movies["title"].astype("string").str.strip()

movies = movies[
    movies["title"].notna() &
    movies["title"].ne("")
].copy()

# 3. Handle missing overviews
movies["overview"] = movies["overview"].fillna("")

# 4. Reset row numbering after cleaning
movies = movies.reset_index(drop=True)

print("Original shape:", df.shape)
print("Cleaned shape:", movies.shape)
print("Remaining duplicate rows:", movies.duplicated().sum())
print("Missing titles:", movies["title"].isna().sum())
print("Missing overviews:", movies["overview"].isna().sum())

Original shape: (45466, 24)
Cleaned shape: (45443, 24)
Remaining duplicate rows: 0
Missing titles: 0
Missing overviews: 0


In [17]:
print(movies[["title", "overview", "genres", "id", "popularity"]].dtypes)

print("\nSample popularity values:")
print(movies["popularity"].head(10).tolist())

print("\nSample movie IDs:")
print(movies["id"].head(10).tolist())

title         string[python]
overview              object
genres                object
id                    object
popularity            object
dtype: object

Sample popularity values:
['21.946943', '17.015539', '11.7129', '3.859495', '8.387519', '17.924927', '6.677277', '2.561161', '5.23158', '14.686036']

Sample movie IDs:
['862', '8844', '15602', '31357', '11862', '949', '11860', '45325', '9091', '710']


In [18]:
# Convert popularity to numeric
movies["popularity"] = pd.to_numeric(
    movies["popularity"],
    errors="coerce"
)

# Keep movie IDs as strings
movies["id"] = movies["id"].astype("string")

print(movies[["title", "id", "popularity"]].dtypes)

print("\nMissing popularity values:", movies["popularity"].isna().sum())
print("\nDataset shape:", movies.shape)

title         string[python]
id            string[python]
popularity           float64
dtype: object

Missing popularity values: 0

Dataset shape: (45443, 24)
